# 42 Audit saved search coverage
CPU only. Reads run 41; no installation or simulator needed. Checks which shorter horizons were fully searched. Preserves original results. Scope and action enumeration remain experimental; outputs are diagnostic, not final dataset labels.


In [ ]:
from pathlib import Path
from google.colab import drive
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import runpy
Path("/content/42_scienceworld_search_coverage_colab.py").write_text('"""Audit saved restricted BFS coverage; no simulator, no new dataset labels."""\nimport json,hashlib\nfrom pathlib import Path\nfrom collections import Counter\nfrom datetime import datetime,timezone\nSOURCE=Path(\'/content/drive/MyDrive/ADNN_Irreversibility/results/scienceworld_fresh_validation_20261107T134439885002Z\')\ndef audit_nodes(nodes,horizon):\n    lookup={tuple(n[\'undo\']):n for n in nodes}\n    assert len(lookup)==len(nodes),\'Duplicate evaluated search sequence\'\n    expected={()};checked=set();missing=[]\n    for depth in range(horizon+1):\n        current=sorted(s for s in expected if len(s)==depth)\n        for seq in current:\n            if seq not in lookup:missing.append(list(seq));continue\n            checked.add(seq);node=lookup[seq]\n            if depth<horizon and not node[\'done\']:\n                assert isinstance(node[\'children\'],list)\n                assert len(node[\'children\'])==len(set(node[\'children\']))\n                expected.update(seq+(a,) for a in node[\'children\'])\n    # Missing ancestors can hide additional descendants; missing count is not total remaining workload.\n    return {\'enumerated_tree_complete\':not missing,\'expected_sequences_known\':len(expected),\n            \'evaluated_expected_sequences\':len(checked),\'missing_frontier_count\':len(missing),\n            \'missing_frontier_sample\':missing[:5],\n            \'limitation\':\'Coverage of saved enumerated actions only; does not independently validate action enumeration or physical state scope.\'}\ndef main():\n    assert SOURCE.is_dir(),\'Mount Drive and check source folder.\'\n    records=json.loads((SOURCE/\'results.json\').read_text());out=SOURCE.parent/(\'scienceworld_coverage_audit_\'+datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S%fZ\'));out.mkdir()\n    results=[];errors=[];hashes={}\n    for record in records:\n        cid=record[\'id\']\n        try:\n            files=list(SOURCE.glob(\'*/\'+cid+\'/nodes.json\'));assert len(files)==1,\'Missing or duplicate node file\'\n            path=files[0];nodes=json.loads(path.read_text());hashes[str(path)]=hashlib.sha256(path.read_bytes()).hexdigest()\n            assert len(nodes)==record[\'nodes_tested\']\n            assert dict(Counter(str(len(n[\'undo\'])) for n in nodes))==record[\'nodes_by_depth\']\n            node_map={tuple(n[\'undo\']):n for n in nodes}\n            for kind,w in record[\'witnesses\'].items():\n                assert w[\'depth\']==len(w[\'undo\']) and w[\'independent_replay_matches\']\n                assert node_map[tuple(w[\'undo\'])][\'kind\']==kind,\'Witness not in evaluated nodes\'\n            horizons={}\n            for h in (1,2,3):\n                c=audit_nodes(nodes,h)\n                valid=record[\'status\'] in (\'budget_exhausted\',\'restricted_search_exhausted\',\'exact_witness_found\')\n                exact=record[\'witnesses\'].get(\'exact\');thermal=record[\'witnesses\'].get(\'thermal\')\n                if not valid:result=\'unresolved_validation_error\'\n                elif exact and exact[\'depth\']<=h:result=\'exact_recovery_witness_under_projection\'\n                elif not c[\'enumerated_tree_complete\']:result=\'unresolved_incomplete_search\'\n                elif thermal and thermal[\'depth\']<=h:result=\'thermal_only_recovery_no_exact_within_restricted_search\'\n                else:result=\'no_exact_or_thermal_recovery_within_restricted_search\'\n                horizons[str(h)]={**c,\'operational_result\':result}\n            r={\'id\':cid,\'source_status\':record[\'status\'],\'source_error\':record.get(\'error\'),\'horizons\':horizons,\'final_dataset_label\':None}\n            results.append(r);print(cid,{h:v[\'operational_result\'] for h,v in horizons.items()},flush=True)\n        except Exception as exc:errors.append({\'id\':cid,\'error\':str(exc)})\n    summary={\'cases_audited\':len(results),\'counts_by_horizon\':{str(h):dict(Counter(r[\'horizons\'][str(h)][\'operational_result\'] for r in results)) for h in (1,2,3)},\n       \'case_validation_errors\':[{\'id\':r[\'id\'],\'error\':r[\'source_error\']} for r in results if r[\'source_status\']==\'validation_error\'],\n       \'audit_errors\':errors,\'final_labels_created\':0,\'scope_coverage_validated\':False,\n       \'note\':\'Repairs conservative horizon reporting using archived branch coverage; no further simulator calls or full-environment irreversibility claims.\',\n       \'saved_to\':str(out)}\n    hashes[str(SOURCE/\'results.json\')]=hashlib.sha256((SOURCE/\'results.json\').read_bytes()).hexdigest()\n    for name,data in [(\'summary.json\',summary),(\'results.json\',results),(\'input_hashes.json\',hashes)]: (out/name).write_text(json.dumps(data,indent=2))\n    (out/\'runner.py\').write_bytes(Path(__file__).read_bytes())\n    print(\'SAVED SEARCH COVERAGE AUDIT COMPLETE\');print(json.dumps(summary,indent=2))\nif __name__==\'__main__\':main()\n')
_ = runpy.run_path("/content/42_scienceworld_search_coverage_colab.py",run_name="__main__")
